# Fine-tuning LoRA - Modele medical (Mission IA experimentale)

**Hackathon TechCorp - Phi-3.5-Financial**

Ce notebook est **experimental / R&D uniquement**, pas de deploiement production.
Il fine-tune `microsoft/Phi-3.5-mini-instruct` en **QLoRA (4-bit)** sur un echantillon
nettoye du dataset HuggingFace `ruslanmv/ai-medical-chatbot` (nettoyage effectue en
Mission DATA, voir `docs/data_quality_medical_report.json` dans le repo).

**A executer sur Colab avec un runtime GPU (Runtime > Modifier le type d'execution > GPU / T4).**

Avertissements importants (repris de `medical_project/Readme.md`) :
- Ce modele ne remplace jamais l'expertise medicale humaine.
- Aucune validation par des professionnels de sante n'a ete faite.
- Usage strictement experimental / demonstration hackathon.

In [ ]:
!pip install -q transformers>=4.45.0 peft>=0.12.0 accelerate>=0.34.0 bitsandbytes>=0.43.0 datasets>=2.20.0

## 1. Charger le dataset nettoye

Uploade les deux fichiers produits par la Mission DATA (dossier `medical_project/` du repo) :
- `medical_dataset_train_sample.json` (4000 exemples question/answer)
- `medical_dataset_test_prompts.json` (15 questions reservees pour les tests conversationnels)

Utilise le widget d'upload ci-dessous, ou monte ton Google Drive si tu preferes.

In [ ]:
from google.colab import files
print("Uploade medical_dataset_train_sample.json et medical_dataset_test_prompts.json")
uploaded = files.upload()

In [ ]:
import json

with open("medical_dataset_train_sample.json", encoding="utf-8") as f:
    train_data = json.load(f)
with open("medical_dataset_test_prompts.json", encoding="utf-8") as f:
    test_prompts = json.load(f)

print(f"Exemples d'entrainement: {len(train_data)}")
print(f"Prompts de test conversationnel: {len(test_prompts)}")
print(train_data[0])

## 2. Charger le modele de base en 4-bit (QLoRA)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

## 3. Configurer LoRA

Configuration alignee sur celle documentee dans `scripts/train_finance_model.py` du repo
(meme rang / memes modules cibles) pour rester coherent avec la methodologie deja utilisee
par l'equipe pour le modele finance.

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["qkv_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Tokeniser le dataset

In [ ]:
from datasets import Dataset

MAX_LENGTH = 512

def format_example(ex):
    return {"text": f"<|user|>\n{ex['question']}<|end|>\n<|assistant|>\n{ex['answer']}<|end|>"}

texts = [format_example(ex) for ex in train_data]
hf_dataset = Dataset.from_list(texts)

def tokenize_fn(examples):
    tok = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)
    tok["labels"] = tok["input_ids"].copy()
    return tok

tokenized_dataset = hf_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.05, seed=42)
print(tokenized_dataset)

## 5. Entrainement

Le `Trainer` journalise la loss a chaque `logging_steps` : on recupere `trainer.state.log_history`
a la fin pour produire la courbe de loss et les metriques a partager (comme demande par les
consignes du hackathon: lien Colab + loss + epochs).

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./medical_lora_adapter",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    fp16=True,
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
)

train_result = trainer.train()

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
train_loss = [(h["step"], h["loss"]) for h in history if "loss" in h]
eval_loss = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

plt.figure()
if train_loss:
    plt.plot(*zip(*train_loss), label="train_loss")
if eval_loss:
    plt.plot(*zip(*eval_loss), label="eval_loss", marker="o")
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Loss d'entrainement - LoRA medical")
plt.legend()
plt.savefig("medical_lora_loss_curve.png")
plt.show()

print(f"Epochs: {training_args.num_train_epochs}")
print(f"Loss finale (train): {train_loss[-1][1] if train_loss else 'n/a'}")
print(f"Loss finale (eval): {eval_loss[-1][1] if eval_loss else 'n/a'}")
print("-> A partager avec l'equipe: lien de ce notebook Colab + cette courbe + les valeurs ci-dessus.")

## 6. Sauvegarder l'adaptateur LoRA

In [ ]:
model.save_pretrained("./medical_lora_adapter")
tokenizer.save_pretrained("./medical_lora_adapter")

import shutil
shutil.make_archive("medical_lora_adapter", "zip", "medical_lora_adapter")

from google.colab import files
files.download("medical_lora_adapter.zip")

## 7. Tests conversationnels (Mission IA - performance conversationnelle)

Quelques echanges de test sur les 15 questions reservees (`medical_dataset_test_prompts.json`,
jamais vues pendant l'entrainement). Compare la reponse du modele fine-tune a la reponse
originale du medecin (reference) pour une evaluation qualitative rapide.

**Rappel: usage experimental seulement, ne pas utiliser ces reponses comme avis medical reel.**

In [ ]:
model.eval()

def generate_answer(question, max_new_tokens=200):
    prompt = f"<|user|>\n{question}<|end|>\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text.strip()

for i, item in enumerate(test_prompts[:6], 1):
    print(f"--- Echange {i} ---")
    print(f"Patient: {item['question'][:300]}")
    print(f"Modele fine-tune: {generate_answer(item['question'])}")
    print(f"Reference (medecin original): {item['reference_answer'][:300]}")
    print()

## Etapes suivantes (Mission CYBER)

Une fois l'adaptateur telecharge, executer localement (avec l'adaptateur charge via `transformers`+`peft`)
le script `scripts/medical_security_bias_tests.py` du repo pour :
- verifier que le modele refuse bien de prescrire des posologies / diagnostics definitifs,
- verifier l'absence de biais notable (genre, origine) sur des paires de prompts symptomes identiques.

Ne pas deployer ce modele en production : c'est un livrable experimental de hackathon.